# Tree accuracy: does the tree agree with the database taxonomy?

One notebook for **all tree variants**, so they can be compared on the same page. It is executed once by the
`tree_accuracy` rule after every tree, PhyloRank run and `assign_taxonomy` has finished. The question: **how well do
the database clade names (phylum ... genus) match the tree, and where they do not, is it a tree problem or a label
problem?** The alignment side (occupancy, divergence, genes) is in `tree_diagnostics`.

Variants are labelled `derep/copies`:

| part | values | meaning |
|---|---|---|
| derep | `all` / `derep` | every genome, or only the dereplicated representatives (external genomes always kept) |
| copies | `single` / `multi` | Complete BUSCO genes only, or also one copy of each Duplicated gene (chosen as described in `copy_selection`) |

A quick-check subset run (label printed below) uses far fewer genomes: do not compare its numbers with a full run.
Trees with fewer genomes have fewer chances to fragment, so a drop in fragmentation between `all` and `derep` is not
by itself evidence of a better tree; the cleaner comparison is `single` vs `multi` on the same genomes.

**How the notebook is organised.** The *Key findings* just below are generated from this run's numbers (computed at the end,
moved up). Each section then shows a figure, a one-line **Takeaway** with the numbers, and how to read the figure. What counts as
OK / worth a look / a problem is set by the thresholds in section 0; change a number there to change a verdict. The closing section
lists what to do about anything that was flagged.

Sections: (1) clade fragmentation, (2) support of the splits that cause it, (3) tips that look misplaced,
(4) RED, (5) can hidden labels be recovered (validation of the naming), (6) how much the variants differ. Monophyly is **independent of where the tree is rooted**
(`scripts/tree_accuracy_lib.py`). Almost everything is a figure; the numbers are also written as TSV.

## 0. Setup

Parameters come from the JSON file named by `NB_PARAMS`, written by the Snakemake rule.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
from Bio import Phylo

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_accuracy_lib as lib
import notebook_narrative as nn

OUT = Path(PARAMS["out_dir"])        # tables and PNG copies of the figures
OUT.mkdir(parents=True, exist_ok=True)
RANKS = lib.RANKS                    # phylum ... species
VARIANTS = [v for v in PARAMS["variants"] if Path(v["tree"]).exists()]
LABELS = [v["label"] for v in VARIANTS]
COLOURS = dict(zip(LABELS, plt.cm.tab10.colors))   # one colour per variant in every figure

print("Run     :", PARAMS["label"])
print("Variants:", ", ".join(LABELS))

### Thresholds

What counts as a problem. Each rule has a *look* value (worth a look) and a *bad* value (a problem). The numbers are defaults for
eukaryotic trees of a few thousand genomes; change them here and every verdict below follows.

In [ ]:
FRAGMENTED = nn.Rule(look=0.10, bad=0.30)        # share of order/family names split into several pieces
SINGLETONS = nn.Rule(look=0.60, bad=1.01)        # share of extra pieces that are a single tip (label or contamination suspects); informative, never a failure by itself
CONFIDENT = 0.95                                 # support at or above this: the tree is confident about the split
CONFIDENT_SHARE = nn.Rule(look=0.80, bad=0.50, higher_is_worse=False)   # share of extra pieces that are confidently placed
MISPLACED = nn.Rule(look=0.01, bad=0.03)         # share of tips that are non-sister misplaced at order/family
WRONG_SIDE = nn.Rule(look=0.20, bad=0.40)        # share of names whose RED is on the wrong side of the neighbouring rank
RED_BELOW = nn.Rule(look=0.005, bad=0.02)        # share of internal nodes below their parent in PhyloRank's raw RED tree
RF_COPIES = nn.Rule(look=0.15, bad=0.35)         # topology distance caused by single vs multi on the same genomes

WRONG_NAME = nn.Rule(look=0.03, bad=0.10)        # share of hidden labels given another existing name (order level)
MIN_TESTED = 30                                  # fewer hidden labels than this: too few to judge
FINDINGS = nn.Findings()
TSA_EXCESS = nn.Rule(look=0.02, bad=0.05)        # how much more often transcriptome (tsa) tips are flagged misplaced than genome tips (difference of shares)
MIN_SOURCE_TIPS = 20                             # fewer transcriptome tips than this: too few to compare

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)


def finish(fig, name):
    '''
    Saves a figure as PNG next to the tables and shows it in the notebook.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        The figure to save.
    name : str
        File name without extension.
    '''
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()


def grouped_bars(axis, table, categories, title, ylabel=""):
    '''
    Draws one group of bars per category with one bar per variant.

    Parameters
    ----------
    axis : matplotlib.axes.Axes
        Axis to draw on.
    table : dict
        {variant label: list of values, one per category}.
    categories : list of str
        Category names (x tick labels).
    title, ylabel : str
        Axis title and y label.
    '''
    width = 0.8 / max(1, len(table))
    for k, label in enumerate(table):
        axis.bar(np.arange(len(categories)) + k * width, table[label], width, color=COLOURS[label], label=label)
    axis.set_xticks(np.arange(len(categories)) + 0.4 - width / 2)
    axis.set_xticklabels(categories)
    axis.set_title(title)
    axis.set_ylabel(ylabel)

## 1. Load the trees and find the clade pieces

If a database name (say an order) is monophyletic, all its tips form **one piece**: one part of the tree that
holds tips of that name and no labelled tips of any other name. Tips of the same name elsewhere form **extra
pieces**. Unlabelled tips (e.g. MAGs) are ignored, so they never count against a name.

In [ ]:
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])   # {genome: {rank: name}} database lineages

TREES, PIECES, SUMMARY, NAMES = {}, {}, {}, {}
for variant in VARIANTS:
    label = variant["label"]
    TREES[label] = Phylo.read(variant["tree"], "newick")
    PIECES[label] = {rank: lib.pieces_unrooted(TREES[label], taxonomy, rank) for rank in RANKS}
    SUMMARY[label], NAMES[label] = lib.fragmentation(TREES[label], taxonomy, pieces_by_rank=PIECES[label])
    n_tips = TREES[label].count_terminals()
    n_labelled = sum(1 for tip in TREES[label].get_terminals() if tip.name in taxonomy)
    print(f"{label:14} {n_tips} tips, {n_labelled} with a database taxonomy")

## 2. Clade fragmentation

Per rank: how many extra pieces there are, how many names are split, and how many tips sit in split names.

**How to read it.**

* A few extra pieces at deep ranks (phylum, class) are usually real: NCBI groups such as Pichiomycetes or Bigyra are
  known to be non-monophyletic.
* Fragmentation that grows toward the tips (many extra pieces at order and family) is more likely wrong labels,
  contaminated genomes or an unstable tree.
* The heatmap lists the most fragmented names with their number of pieces in every variant, so you can see which
  clades a genome set or gene set changes.

In [ ]:
fragmentation = pl.concat([pl.DataFrame(SUMMARY[label]).with_columns(pl.lit(label).alias("variant")) for label in LABELS])
fragmentation.write_csv(OUT / "fragmentation.tsv", separator="\t")
by_rank = lambda col: {label: fragmentation.filter(pl.col("variant") == label).sort(
    pl.col("rank").replace_strict(RANKS, list(range(len(RANKS)))))[col].to_list() for label in LABELS}

fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
grouped_bars(axes[0], by_rank("extra_pieces"), RANKS, "extra pieces (beyond one per name)")
grouped_bars(axes[1], by_rank("fragmented_names"), RANKS, "names split into several pieces")
grouped_bars(axes[2], by_rank("tips_in_fragmented"), RANKS, "tips in split names")
axes[0].legend(fontsize=8)
finish(fig, "fragmentation_by_rank")

frag_share = {}
for label in LABELS:
    rows = {r["rank"]: r for r in SUMMARY[label]}
    frag_share[label] = max(rows[k]["fragmented_names"] / max(rows[k]["names"], 1) for k in ("order", "family") if k in rows)
s_f, t_f = nn.across(FRAGMENTED, frag_share, nn.pct)
deep = {l: {r["rank"]: r["fragmented_names"] for r in SUMMARY[l]} for l in LABELS}
FINDINGS.say(s_f, f"{t_f} of order or family names are split into several pieces in the tree "
                  f"(at phylum level {min(d.get('phylum', 0) for d in deep.values())}-{max(d.get('phylum', 0) for d in deep.values())} names).",
             evidence=f"fragmented share {t_f}",
             action="Look at the most fragmented names (next figure) and at whether the extra pieces are single tips (labels or contamination) or real clades.")

In [ ]:
N_SHOWN = 25
pieces_table = (pl.concat([pl.DataFrame(NAMES[label]).select("rank", "name", "tips", "pieces")
                           .with_columns(pl.lit(label).alias("variant")) for label in LABELS])
                .pivot(on="variant", index=["rank", "name"], values="pieces").fill_null(0))
pieces_table = pieces_table.with_columns(
    pl.max_horizontal(LABELS).alias("max_pieces"),
    (pl.max_horizontal(LABELS) - pl.min_horizontal(LABELS)).alias("spread"))
pieces_table.write_csv(OUT / "pieces_by_name.tsv", separator="\t")

top = pieces_table.filter(pl.col("rank").is_in(["class", "order", "family"])).sort("max_pieces", descending=True).head(N_SHOWN)
grid = top.select(LABELS).to_numpy()
fig, axis = plt.subplots(figsize=(1.0 * len(LABELS) + 4, 0.28 * N_SHOWN + 1.5))
image = axis.imshow(grid, aspect="auto", cmap="magma_r")
axis.set_xticks(range(len(LABELS)))
axis.set_xticklabels(LABELS, rotation=45, ha="right")
axis.set_yticks(range(top.height))
axis.set_yticklabels([f"{r[0]}  {n}" for r, n in zip(top["rank"], top["name"])], fontsize=8)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        axis.text(j, i, int(grid[i, j]), ha="center", va="center", fontsize=7,
                  color="white" if grid[i, j] > grid.max() / 2 else "black")
plt.colorbar(image, label="pieces")
axis.set_title(f"{N_SHOWN} most fragmented names (class/order/family): pieces per variant")
finish(fig, "most_fragmented_names")

single_tip = {}
for label in LABELS:
    sizes = [s for n in NAMES[label] if n["rank"] in ("order", "family") for s in n["extra_sizes"]]
    single_tip[label] = (sum(s == 1 for s in sizes) / len(sizes)) if sizes else 0.0
s_s, t_s = nn.across(SINGLETONS, single_tip, nn.pct)
worst_name = top.row(0, named=True) if top.height else None
FINDINGS.say(s_s, f"{t_s} of the extra order/family pieces are a single tip, so most fragmentation is lone genomes sitting outside their clade (wrong label or contamination) rather than clades split in two."
                  + (f" The most fragmented name is `{worst_name['name']}` ({worst_name['rank']}, up to {worst_name['max_pieces']} pieces)." if worst_name else ""),
             evidence=f"single-tip share {t_s}",
             action="Check the labels and BUSCO duplication of the lone tips (see the misplaced-tip genome checks below).")

## 3. Are the splits that fragment a clade supported?

Each extra piece hangs below a branch. Its **support** is FastTree's SH-like value on that branch (0-1).

* **high (>= 0.95)** - the data confidently place those tips elsewhere. The disagreement with the database is real
  (a wrong label, a contaminated genome or biology) and the tree is not the problem.
* **low** - the placement is uncertain and the tree may simply be wrong there.

SH-like values are generous, so read high support as "not obviously noise", not as proof.

In [ ]:
support_rows = [{"variant": label, "rank": rank, "name": name, "piece_tips": size, "support": support}
                for label in LABELS for rank in RANKS
                for name, size, support in lib.extra_piece_supports(PIECES[label][rank])]
extra_support = pl.DataFrame(support_rows, schema={"variant": pl.Utf8, "rank": pl.Utf8, "name": pl.Utf8,
                                                    "piece_tips": pl.Int64, "support": pl.Float64})
extra_support.write_csv(OUT / "extra_piece_support.tsv", separator="\t")
with_support = extra_support.drop_nulls("support")

if with_support.height:
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
    for label in LABELS:
        sub = with_support.filter(pl.col("variant") == label)
        axes[0].hist(sub["support"], bins=20, range=(0, 1), histtype="step", linewidth=1.6, color=COLOURS[label], label=label)
        axes[1].scatter(sub["piece_tips"], sub["support"], s=8, alpha=0.5, color=COLOURS[label])
    axes[0].set_title("support of the branch above each extra piece")
    axes[0].legend(fontsize=8)
    axes[1].set_xscale("log")
    axes[1].set_xlabel("piece size (tips)")
    axes[1].set_ylabel("support")
    shares = {label: [(with_support.filter(pl.col("variant") == label)["support"] < 0.8).mean(),
                      (with_support.filter(pl.col("variant") == label)["support"] >= 0.95).mean()] for label in LABELS}
    grouped_bars(axes[2], shares, ["support < 0.8", "support >= 0.95"], "share of extra pieces")
    finish(fig, "extra_piece_support")
else:
    print("No support values in these trees.")

if with_support.height:
    confident = {l: float((with_support.filter(pl.col("variant") == l)["support"] >= CONFIDENT).mean()) for l in LABELS
                 if with_support.filter(pl.col("variant") == l).height}
    s_c, t_c = nn.across(CONFIDENT_SHARE, confident, nn.pct)
    FINDINGS.say(s_c, f"{t_c} of the extra pieces hang below a branch with support of at least {CONFIDENT:.2f}: the tree is confident about the split, "
                      "so the disagreement with the database is real (a wrong label, a contaminated genome or biology) and not tree noise.",
                 evidence=f"confident share {t_c}",
                 action="Low-support splits are the ones the tree itself doubts: add genes or genomes there before changing any label.")

## 4. Tips that look misplaced

A tip is flagged at a rank when its nearest labelled neighbours (at least 8) **agree at >= 90% on a different name**.
`sister = True` means that name belongs to the same parent clade (for example two orders of one class), a milder
disagreement that is often just a classification difference. The counts below separate the two; the genome checks
use only the non-sister tips.

Flagged genomes are then compared with all genomes on BUSCO completeness, BUSCO duplication (a high fraction can mean
contamination or a mixed assembly) and whether their database taxonomy is only `partial`.

**How to read it.** If flagged genomes look like everyone else, the cause is most likely the label (or real
biology). If they are less complete, more duplicated or more often `partial`, look at the genomes first.

In [ ]:
RANKS_FLAGGED = ["phylum", "class", "order", "family"]
flagged = {label: {rank: lib.misplaced_tips(TREES[label], taxonomy, rank) for rank in RANKS_FLAGGED} for label in LABELS}

misplaced = pl.DataFrame(
    [{"variant": label, "rank": rank, **row} for label in LABELS for rank in RANKS_FLAGGED for row in flagged[label][rank]],
    schema={"variant": pl.Utf8, "rank": pl.Utf8, "tip": pl.Utf8, "label": pl.Utf8, "neighbours_say": pl.Utf8,
            "agreement": pl.Float64, "n_neighbours": pl.Int64, "sister": pl.Boolean})
misplaced.write_csv(OUT / "misplaced_tips.tsv", separator="\t")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
grouped_bars(axes[0], {l: [sum(not r["sister"] for r in flagged[l][k]) for k in RANKS_FLAGGED] for l in LABELS},
             RANKS_FLAGGED, "non-sister misplaced tips", "tips")
grouped_bars(axes[1], {l: [sum(r["sister"] for r in flagged[l][k]) for k in RANKS_FLAGGED] for l in LABELS},
             RANKS_FLAGGED, "sister-group swaps", "tips")
axes[0].legend(fontsize=8)
finish(fig, "misplaced_counts")

tips_total = {l: TREES[l].count_terminals() for l in LABELS}
mis_share = {l: len({r["tip"] for rank in ("order", "family") for r in flagged[l][rank] if not r["sister"]}) / tips_total[l] for l in LABELS}
s_m, t_m = nn.across(MISPLACED, mis_share, lambda v: nn.pct(v, 1))
FINDINGS.say(s_m, f"{t_m} of tips look misplaced at order or family level (their nearest labelled neighbours agree on another, non-sister name).",
             evidence=f"misplaced share {t_m}",
             action="Compare the flagged genomes with everyone else in the genome checks below before assuming the label is wrong.")

In [ ]:
# tax_status (complete / partial / unresolved) and the BUSCO full table of every genome
mapping = pl.read_csv(PARAMS["taxonomy_mapping"], separator="\t").select("acc", "tax_status")
tax_status = dict(zip(mapping["acc"], mapping["tax_status"]))
busco_tables = {}
for entry in PARAMS["busco_manifests"]:
    manifest, base = Path(entry["manifest"]), Path(entry["base"])
    if manifest.exists():
        qc = pl.read_csv(manifest, separator="\t")
        busco_tables.update({a: Path(r) if Path(r).is_absolute() else base / r
                             for a, r in zip(qc["genome_acc"], qc["busco_result"])})

_cache = {}
def busco_stats(accession):
    '''
    BUSCO completeness and duplication of one genome (cached across variants).

    Parameters
    ----------
    accession : str
        Genome accession (tree tip name).

    Returns
    -------
    tuple or None
        (Complete count, Duplicated count, Duplicated fraction), or None without a BUSCO table.
    '''
    if accession not in _cache:
        path = busco_tables.get(accession)
        if path is None or not Path(path).exists():
            _cache[accession] = None
        else:
            counts = lib.busco_status_counts(path)
            total = sum(counts.values()) or 1
            _cache[accession] = (counts.get("Complete", 0), counts.get("Duplicated", 0), counts.get("Duplicated", 0) / total)
    return _cache[accession]

In [ ]:
def group_values(accessions):
    '''Lists of completeness, duplication fraction and partial-taxonomy flags for a set of genomes.'''
    stats = [s for s in map(busco_stats, accessions) if s]
    return ([s[0] for s in stats], [s[2] for s in stats],
            [tax_status.get(a) == "partial" for a in accessions])


fig, axes = plt.subplots(1, 3, figsize=(17, 4))
positions, data_c, data_d, ticks, tick_pos, partial = [], [], [], [], [], []
for k, label in enumerate(LABELS):
    odd = sorted({r["tip"] for rank in ("order", "family") for r in flagged[label][rank] if not r["sister"]})
    everyone = [t.name for t in TREES[label].get_terminals()]
    for j, (group, name) in enumerate(((odd, "misplaced"), (everyone, "all tips"))):
        c, d, p = group_values(group)
        pos = k * 3 + j
        positions.append(pos); data_c.append(c or [np.nan]); data_d.append(d or [np.nan])
        ticks.append(f"{label}\n{name} ({len(group)})"); tick_pos.append(pos)
        partial.append(float(np.mean(p)) if p else 0.0)
colours = [COLOURS[LABELS[p // 3]] if p % 3 == 0 else "lightgrey" for p in positions]
for axis, data, title in ((axes[0], data_c, "BUSCO Complete genes"), (axes[1], data_d, "BUSCO Duplicated fraction")):
    box = axis.boxplot(data, positions=positions, widths=0.8, patch_artist=True, showfliers=False)
    for patch, colour in zip(box["boxes"], colours):
        patch.set_facecolor(colour)
    axis.set_xticks(tick_pos); axis.set_xticklabels(ticks, rotation=60, ha="right", fontsize=7)
    axis.set_title(title)
axes[2].bar(positions, partial, color=colours)
axes[2].set_xticks(tick_pos); axes[2].set_xticklabels(ticks, rotation=60, ha="right", fontsize=7)
axes[2].set_title("share with only a partial database taxonomy")
finish(fig, "misplaced_genome_quality")

first = LABELS[0]
odd_first = sorted({r["tip"] for rank in ("order", "family") for r in flagged[first][rank] if not r["sister"]})
c_odd, d_odd, p_odd = group_values(odd_first)
c_all, d_all, p_all = group_values([t.name for t in TREES[first].get_terminals()])
reasons = []
if c_odd and c_all:
    if np.median(c_odd) < 0.9 * np.median(c_all):
        reasons.append("They are less complete than average: check the genomes before the labels.")
    if np.median(d_odd) > np.median(d_all) + 0.02:
        reasons.append("They are more duplicated than average, which can mean contamination or a mixed assembly.")
    if np.mean(p_odd) > np.mean(p_all) + 0.10:
        reasons.append("They are more often only partially labelled, so the database name itself may be incomplete.")
    if not reasons:
        reasons.append("They look like everyone else, so the cause is more likely the label or real biology.")
    nn.show(f"**Takeaway.** In `{first}` the {len(odd_first)} misplaced genomes have a median of {np.median(c_odd):.0f} Complete BUSCO genes against {np.median(c_all):.0f} for all tips, "
            f"a median duplicated fraction of {np.median(d_odd):.2f} against {np.median(d_all):.2f}, and {nn.pct(np.mean(p_odd))} partial taxonomies against {nn.pct(np.mean(p_all))}. "
            + " ".join(reasons))

Tips flagged as non-sister misplaced at order or family level in the most variants. A tip flagged in every
variant is a robust problem (a label, contamination, or real biology); one that appears in a single variant is
sensitive to the genome or gene set.

In [ ]:
N_TIPS = 25
flag_counts = (misplaced.filter(~pl.col("sister") & pl.col("rank").is_in(["order", "family"]))
               .group_by("tip").agg(pl.col("variant").n_unique().alias("variants"),
                                    pl.col("label").first(), pl.col("neighbours_say").first()))
shown = flag_counts.sort(["variants", "tip"], descending=[True, False]).head(N_TIPS)
if shown.height:
    grid = np.array([[any(r["tip"] == tip and not r["sister"] for rank in ("order", "family")
                          for r in flagged[l][rank]) for l in LABELS] for tip in shown["tip"]], dtype=float)
    fig, axis = plt.subplots(figsize=(1.0 * len(LABELS) + 5, 0.28 * shown.height + 1.5))
    axis.imshow(grid, aspect="auto", cmap="Reds", vmin=0, vmax=1)
    axis.set_xticks(range(len(LABELS)))
    axis.set_xticklabels(LABELS, rotation=45, ha="right")
    axis.set_yticks(range(shown.height))
    axis.set_yticklabels([f"{t}  {a} -> {b}" for t, a, b in zip(shown["tip"], shown["label"], shown["neighbours_say"])], fontsize=7)
    axis.set_title(f"top {shown.height} non-sister misplaced tips (order/family); red = flagged")
    finish(fig, "misplaced_tips_by_variant")
else:
    print("No non-sister misplaced tips.")

robust = flag_counts.filter(pl.col("variants") == len(LABELS))
FINDINGS.say(nn.LOOK if robust.height else nn.OK,
             f"{robust.height} tip(s) are flagged in every variant" + (": " + ", ".join(f"`{t}` ({a} -> {b})" for t, a, b in robust.head(3).select("tip", "label", "neighbours_say").iter_rows()) if robust.height else "") + ".",
             evidence=f"{robust.height} robust tips",
             action="A tip flagged in every variant is a robust problem (label, contamination or real biology): inspect these first.")

### Misplaced tips by source: genomes against transcriptomes

Tips come from different sources (`source_group` in the tree metadata: `fungi`, `algae`, `p10k`, `ncbi`, `external`, and `tsa` for
transcriptome shotgun assemblies). A transcriptome is less trustworthy than a genome (isoforms, contamination, a partial gene set),
so this compares how often each source's tips are flagged as non-sister misplaced at order or family level. If `tsa` tips are
flagged far more often than genome tips, treat them as lower-confidence placements (or drop them from the tree) before trusting
a surprising result. Sources that are not in a BUSCO quality-control manifest appear as `unknown`.

In [ ]:
import source_groups as sg

SOURCE_GROUPS = sg.read_source_groups([e["manifest"] for e in PARAMS["busco_manifests"]])
rows = []
for label in LABELS:
    tips = [t.name for t in TREES[label].get_terminals()]
    odd = {r["tip"] for rank in ("order", "family") for r in flagged[label][rank] if not r["sister"]}
    rows += [{"variant": label, **row} for row in sg.by_source(tips, SOURCE_GROUPS, odd)]
by_source = pl.DataFrame(rows, schema={"variant": pl.Utf8, "group": pl.Utf8, "tips": pl.Int64, "flagged": pl.Int64, "share": pl.Float64})
by_source.write_csv(OUT / "misplaced_by_source.tsv", separator="\t")
show_all(by_source.pivot(on="variant", index="group", values=["tips", "share"]).sort("group"))

GROUPS_SHOWN = sorted(by_source["group"].unique().to_list())
fig, axis = plt.subplots(figsize=(1.6 * len(GROUPS_SHOWN) + 4, 3.8))
share_table = {l: [next((r["share"] for r in rows if r["variant"] == l and r["group"] == g), 0.0) for g in GROUPS_SHOWN] for l in LABELS}
grouped_bars(axis, share_table, GROUPS_SHOWN, "tips flagged as non-sister misplaced (order/family), by source", "share of the source's tips")
axis.legend(fontsize=8)
finish(fig, "misplaced_by_source")

tsa_gap = {}
for label in LABELS:
    mine = [r for r in rows if r["variant"] == label]
    t, g = [r for r in mine if r["group"] == "tsa"], [r for r in mine if r["group"] != "tsa"]
    if t and sum(r["tips"] for r in t) >= MIN_SOURCE_TIPS and g:
        tsa_gap[label] = (sum(r["flagged"] for r in t) / sum(r["tips"] for r in t)
                          - sum(r["flagged"] for r in g) / sum(r["tips"] for r in g))
if tsa_gap:
    s_t, t_t = nn.across(TSA_EXCESS, tsa_gap, lambda v: f"{100 * v:+.1f} points")
    FINDINGS.say(s_t, f"The share of transcriptome (`tsa`) tips flagged as misplaced differs from that of genome tips by {t_t} (positive = transcriptomes are worse).",
                 evidence=f"tsa minus genome flagged share {t_t}",
                 action="If transcriptomes are flagged much more often, check their BUSCO duplication (isoforms, contamination) or leave them out of the tree.")
else:
    print(f"Fewer than {MIN_SOURCE_TIPS} transcriptome (tsa) tips in these trees: no comparison made.")

## 5. RED (relative evolutionary divergence)

PhyloRank gives every node a RED value between 0 (root) and 1 (tips); `assign_taxonomy` uses RED cutoffs per rank to
decide how deep a database name may be inherited by the tips below a node. The cutoffs are the median RED of the
existing names at each rank **on that tree**, so they differ between variants.

**How to read it.**

* *Spread*: boxes are the RED of each named taxon at a rank, the red mark is the cutoff. If boxes of neighbouring
  ranks overlap a lot, one cutoff cannot separate the ranks. The `wrong side` panel gives the share of names whose
  RED lies beyond the neighbouring rank's median: small is good; 10-20% means RED only roughly follows the database
  ranks (expected, as NCBI ranks are not equal in age across eukaryotes).
* *Monotonicity*: PhyloRank's RED is a median over several rootings, so a child can end up below its parent, which
  also gives negative branch lengths in the RED tree. The `phylorank_red` rule therefore raises each node's
  RED to at least its parent's and rewrites the published RED trees (`negative_branches_published` must be 0); the
  bars show PhyloRank's raw tree (`*.phylorank_raw.tree`). A handful of nodes is fine; many would mean the tree is poorly rooted or unstable near
  the root.

In [ ]:
RED_RANKS = ["phylum", "class", "order", "family", "genus"]
red = {v["label"]: lib.red_by_rank(v["red_tsv"], v["red_dict"]) for v in VARIANTS}

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
width = 0.8 / len(LABELS)
for k, label in enumerate(LABELS):
    for i, rank in enumerate(RED_RANKS):
        cutoff, values = red[label].get(rank, (None, []))
        if not values:
            continue
        pos = i + k * width
        box = axes[0].boxplot([values], positions=[pos], widths=width * 0.85, patch_artist=True, showfliers=False)
        box["boxes"][0].set_facecolor(COLOURS[label])
        axes[0].plot([pos - width * 0.4, pos + width * 0.4], [cutoff, cutoff], "r-", linewidth=2)
axes[0].set_xticks(np.arange(len(RED_RANKS)) + 0.4 - width / 2)
axes[0].set_xticklabels(RED_RANKS)
axes[0].set_title("RED of each named taxon (red mark = cutoff)")
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=COLOURS[l], label=l) for l in LABELS], fontsize=8)
overlap = {label: lib.rank_overlap(red[label]) for label in LABELS}
grouped_bars(axes[1], {l: [np.nanmean(overlap[l].get(r, (np.nan, np.nan))) for r in RED_RANKS] for l in LABELS},
             RED_RANKS, "share of names on the wrong side of the neighbouring rank")
rows = []
for variant in VARIANTS:
    # the published red_tree is made monotone after PhyloRank; the untouched output is kept next to it
    final = Path(variant["red_tree"])
    raw = final.with_name(final.stem + ".phylorank_raw" + final.suffix)
    n_nodes, n_below, n_negative, worst = lib.red_monotonicity(Phylo.read(raw if raw.exists() else final, "newick"))
    _, _, n_negative_final, _ = lib.red_monotonicity(Phylo.read(final, "newick"))
    rows.append({"variant": variant["label"], "internal_nodes": n_nodes, "nodes_below_parent": n_below,
                 "negative_branches": n_negative, "most_negative": worst,
                 "negative_branches_published": n_negative_final})
monotonicity = pl.DataFrame(rows)
x = np.arange(len(LABELS))
axes[2].bar(x - 0.2, monotonicity["nodes_below_parent"], 0.4, label="nodes below their parent")
axes[2].bar(x + 0.2, monotonicity["negative_branches"], 0.4, label="negative branches")
axes[2].set_xticks(x); axes[2].set_xticklabels(LABELS, rotation=30, ha="right")
axes[2].set_title("non-monotone RED in PhyloRank's raw tree")
axes[2].legend(fontsize=8)
finish(fig, "red")

monotonicity.write_csv(OUT / "red_monotonicity.tsv", separator="\t")
pl.DataFrame([{"variant": l, "rank": r, "cutoff": red[l][r][0], "taxa": len(red[l][r][1]),
               "median": float(np.median(red[l][r][1])) if red[l][r][1] else None}
              for l in LABELS for r in red[l]]).write_csv(OUT / "red_summary.tsv", separator="\t")

wrong = {l: float(np.nanmean([np.nanmean(overlap[l].get(r, (np.nan, np.nan))) for r in RED_RANKS])) for l in LABELS}
s_wr, t_wr = nn.across(WRONG_SIDE, wrong, nn.pct)
below = {r["variant"]: r["nodes_below_parent"] / r["internal_nodes"] for r in monotonicity.iter_rows(named=True)}
s_bl, t_bl = nn.across(RED_BELOW, below, lambda v: nn.pct(v, 2))
published = int(monotonicity["negative_branches_published"].sum()) if "negative_branches_published" in monotonicity.columns else 0
raw_neg = int(monotonicity["negative_branches"].sum())
FINDINGS.say(s_wr, f"On average {t_wr} of names have a RED beyond the neighbouring rank's median, so RED follows the database ranks only roughly (10-20% is expected, as NCBI ranks differ in age).",
             evidence=f"wrong-side share {t_wr}", action="If this is high, one RED cutoff per rank cannot separate the ranks; treat new rank names with caution.")
FINDINGS.say(nn.BAD if published else s_bl,
             (f"The published RED trees still have {published} negative branches." if published else
              f"PhyloRank's raw RED tree had {raw_neg} negative branches ({t_bl} of internal nodes below their parent); the published trees have none because every node is raised to at least its parent's RED."),
             evidence=f"raw negative {raw_neg}, published {published}",
             action="Published negatives mean `monotone_red_tree.py` did not run: re-run the phylorank_red rule." if published else
                    "Many raw violations would mean the tree is poorly rooted or unstable near the root.")

## 6. Can hidden labels be recovered? (validation of the naming)

Everything above compares the tree with the database. This section asks whether **the naming method itself** is trustworthy.
In the `label_masking` rule a random share of the labelled genomes (default 15%) has its label hidden, `assign_taxonomy` names
them like any new genome (from the tree and the remaining labels), and the result is compared with the hidden truth. Each answer
is one of:

* **correct** - the hidden name came back;
* **novel** - a placeholder name (the method says "new taxon"), which is the right answer only when the taxon has no other
  member; hidden genomes whose true name has no visible member left are counted separately (`unseen`) and not scored;
* **wrong** - another existing database name. This is the costly error: it is what the majority-name rule for MIXED nodes and
  inheritance from neighbours can produce. It never merges database names (a check in `assign_taxonomy` enforces that), it
  only mis-names *unlabelled* genomes.

Five replicates with different masks give a spread. At species level most names are seen once, so little can be scored there.
The second table lists the MIXED nodes that were named by their majority (`mixed_nodes.tsv`): these are the places the rule acts.

In [ ]:
mask_frames = []
for v in VARIANTS:
    if Path(v["masking"]).exists():
        mask_frames.append(pl.read_csv(v["masking"], separator="\t").with_columns(pl.lit(v["label"]).alias("variant")))
if not mask_frames:
    nn.show("**Takeaway.** No `label_masking.tsv` found: run the `label_masking` rule.")
else:
    masking = pl.concat(mask_frames)
    by_rank = (masking.group_by("variant", "rank").agg(pl.col(c).sum() for c in ("tested", "correct", "novel", "wrong", "unseen"))
               .with_columns((pl.col("correct") / pl.col("tested")).alias("accuracy"), (pl.col("wrong") / pl.col("tested")).alias("wrong_rate")))
    by_rank.write_csv(OUT / "label_masking_summary.tsv", separator="\t")
    spread = (masking.with_columns((pl.col("wrong") / pl.col("tested")).alias("w"))
              .group_by("variant", "rank").agg(pl.col("w").min().alias("w_min"), pl.col("w").max().alias("w_max")))
    ranks_ordered = [r for r in RANKS if r != "domain"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
    grouped_bars(axes[0], {l: [float(by_rank.filter((pl.col("variant") == l) & (pl.col("rank") == r))["accuracy"].fill_null(0).sum()) for r in ranks_ordered]
                            for l in LABELS if l in by_rank["variant"].to_list()}, ranks_ordered, "hidden label recovered", "share")
    grouped_bars(axes[1], {l: [float(by_rank.filter((pl.col("variant") == l) & (pl.col("rank") == r))["wrong_rate"].fill_null(0).sum()) for r in ranks_ordered]
                            for l in LABELS if l in by_rank["variant"].to_list()}, ranks_ordered, "hidden label replaced by another existing name", "share")
    axes[0].legend(fontsize=8)
    finish(fig, "label_masking")
    order_rows = by_rank.filter(pl.col("rank") == "order")
    enough = order_rows.filter(pl.col("tested") >= MIN_TESTED)
    if enough.height:
        wrong_o = {r["variant"]: r["wrong_rate"] for r in enough.iter_rows(named=True)}
        acc_o = {r["variant"]: r["accuracy"] for r in enough.iter_rows(named=True)}
        s_wo, t_wo = nn.across(WRONG_NAME, wrong_o, nn.pct)
        _, t_ao = nn.across(nn.Rule(1, 2), acc_o, nn.pct)
        span = spread.filter(pl.col("rank") == "order")
        lo, hi = float(span["w_min"].min()), float(span["w_max"].max())
        FINDINGS.say(s_wo, f"When 15% of labels are hidden, {t_ao} of them come back at order level and {t_wo} are given another existing order name "
                           f"(replicates range {nn.pct(lo, 1)} to {nn.pct(hi, 1)}); the rest are called novel.",
                     evidence=f"order accuracy {t_ao}, wrong {t_wo}",
                     action="Wrong names concentrate where the tree disagrees with the labels (fragmented names, MIXED nodes); compare them with the table of MIXED nodes below.")
    else:
        FINDINGS.say(nn.LOOK, f"Fewer than {MIN_TESTED} hidden labels could be scored at order level; the masking result is too thin to judge.",
                     evidence="few scored labels", action="Run on a larger tree.")
    show_all(by_rank.sort("variant", "rank").select("variant", "rank", "tested", "correct", "novel", "wrong", "unseen",
                                                    pl.col("accuracy").round(3), pl.col("wrong_rate").round(3)))

In [ ]:
mixed_frames = [pl.read_csv(v["mixed_nodes"], separator="\t").with_columns(pl.lit(v["label"]).alias("variant"))
                for v in VARIANTS if Path(v["mixed_nodes"]).exists() and Path(v["mixed_nodes"]).stat().st_size > 20]
if mixed_frames:
    mixed = pl.concat(mixed_frames)
    mixed.write_csv(OUT / "mixed_nodes_all.tsv", separator="\t")
    per = (mixed.group_by("variant", "rank").agg(pl.len().alias("nodes"), pl.col("n_reference").sum().alias("reference_leaves"),
                                                  (pl.col("majority_count") / pl.col("n_reference")).median().alias("median_majority_share"))
           .sort("variant", "rank"))
    show_all(per.with_columns(pl.col("median_majority_share").round(2)))
    close = mixed.filter(pl.col("majority_count") / pl.col("n_reference") < 0.6)
    first = LABELS[0]
    n_first = mixed.filter(pl.col("variant") == first).height
    nn.show(f"**Takeaway.** In `{first}` {n_first} MIXED nodes were named by their majority; in {close.filter(pl.col('variant') == first).height} of them the majority holds "
            "under 60% of the reference genomes, so a new genome placed there is close to a coin flip between names. The pattern to look for is "
            "a large clade (hundreds of genomes) whose labels span several classes: that is a database inconsistency, not something the naming can fix.")
else:
    nn.show("No MIXED nodes were named by majority in these trees.")

## 7. How different are the variants?

Normalised **Robinson-Foulds** distance on the tips two trees share: the share of branches found in one tree but not
the other (0 = identical topology, 1 = nothing in common).

**How to read it.** Compare the distance caused by one choice with the distance caused by another. If `single` vs
`multi` (same genomes) is much smaller than `all` vs `derep` (different genomes), the Duplicated genes barely change
the tree and the genome set matters more. A few percent is what tree-search noise alone gives, so only large
differences are meaningful.

In [ ]:
rf = np.zeros((len(LABELS), len(LABELS)))
for i, first in enumerate(LABELS):
    for j, second in enumerate(LABELS):
        if i < j:
            rf[i, j] = rf[j, i] = lib.rf_distance(TREES[first], TREES[second])[1]
pl.DataFrame({"tree": LABELS, **{l: np.round(rf[:, k], 4) for k, l in enumerate(LABELS)}}).write_csv(
    OUT / "rf_matrix.tsv", separator="\t")

fig, axis = plt.subplots(figsize=(1.0 * len(LABELS) + 3, 1.0 * len(LABELS) + 2))
image = axis.imshow(rf, vmin=0, cmap="viridis")
axis.set_xticks(range(len(LABELS))); axis.set_xticklabels(LABELS, rotation=45, ha="right")
axis.set_yticks(range(len(LABELS))); axis.set_yticklabels(LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        axis.text(j, i, f"{rf[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8)
plt.colorbar(image, label="normalised Robinson-Foulds distance")
axis.set_title("topological distance between variants")
finish(fig, "rf_matrix")

def _pair(a, b):
    return rf[LABELS.index(a), LABELS.index(b)] if a in LABELS and b in LABELS else None
copies_d = [d for d in (_pair(f"{x}/single", f"{x}/multi") for x in ("all", "derep")) if d is not None]
derep_d = [d for d in (_pair(f"all/{c}", f"derep/{c}") for c in ("single", "multi")) if d is not None]
if copies_d:
    value = float(np.mean(copies_d))
    FINDINGS.say(RF_COPIES.status(value),
                 f"Choosing single vs multi copy moves {nn.pct(value, 1)} of branches"
                 + (f", against {nn.pct(float(np.mean(derep_d)), 1)} for choosing all vs dereplicated genomes" if derep_d else "")
                 + ". " + ("The Duplicated genes barely change the tree." if value < float(np.mean(derep_d or [1])) else "The Duplicated genes change the tree more than the genome set does."),
                 evidence=f"RF single/multi {value:.3f}",
                 action="If single and multi differ a lot, prefer single-copy (no paralog risk) and check `copy_selection` for the genes behind the difference.")

## 8. Summary and what to do next

The table has one row per variant. Below it, every finding that was not fine, worst first, with the evidence and a suggested
action. Tables and PNG copies of every figure are written next to the executed notebook (`tables/tree_accuracy/`).

In [ ]:
summary = pl.DataFrame([{
    "variant": label,
    "tips": TREES[label].count_terminals(),
    "extra_pieces_order": [r for r in SUMMARY[label] if r["rank"] == "order"][0]["extra_pieces"],
    "extra_pieces_family": [r for r in SUMMARY[label] if r["rank"] == "family"][0]["extra_pieces"],
    "extra_pieces_support>=0.95": round(float((with_support.filter(pl.col("variant") == label)["support"] >= 0.95).mean()), 3)
                                  if with_support.height else None,
    "non_sister_misplaced_order": sum(not r["sister"] for r in flagged[label]["order"]),
    "red_nodes_below_parent": int(monotonicity.filter(pl.col("variant") == label)["nodes_below_parent"][0]),
} for label in LABELS])
show_all(summary)

nn.show("### What to do next\n\n" + FINDINGS.actions())
pl.DataFrame(FINDINGS.table()).write_csv(OUT / "findings.tsv", separator="\t")

## Key findings

In [ ]:
nn.show(FINDINGS.verdict() + "\n\n" + FINDINGS.bullets() +
        "\n\n*Generated from this run's numbers; the thresholds are in section 0. Details and figures follow.*")